<a href="https://colab.research.google.com/github/gredy/ZTM_RAG/blob/main/Copy_of_NAIVE_RAG_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================================
# NAIVE RAG MODEL IMPLEMENTATION
# ======================================

# Install packages
! pip install sentence-transformers faiss-cpu transformers torch pandas

import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import pipeline

# --------------------------------------
# Step 1: Example Corpus
# --------------------------------------

documents = [
    "Retrieval-Augmented Generation combines retrieval and generation.",
    "Large Language Models may hallucinate facts.",
    "FAISS is a library for efficient similarity search.",
    "RAG improves factual accuracy by retrieving external knowledge.",
    "Sentence Transformers convert text into dense embeddings."
]

# --------------------------------------
# Step 2: Embedding Model
# --------------------------------------

embedding_model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

doc_embeddings = embedding_model.encode(documents)

# --------------------------------------
# Step 3: Create Vector Index
# --------------------------------------

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(doc_embeddings))

# --------------------------------------
# Step 4: Generator Model
# --------------------------------------

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

# --------------------------------------
# Step 5: Retrieval Function
# --------------------------------------

def retrieve(query, top_k=2):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    retrieved_docs = [documents[i] for i in indices[0]]

    return retrieved_docs

# --------------------------------------
# Step 6: RAG Pipeline
# --------------------------------------

def rag_answer(query):

    retrieved_docs = retrieve(query)

    context = " ".join(retrieved_docs)

    prompt = f"""
    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    answer = generator(
        prompt,
        max_length=100
    )[0]['generated_text']

    return answer, retrieved_docs

# --------------------------------------
# Step 7: Example
# --------------------------------------

question = "Why does RAG reduce hallucinations?"

response, docs = rag_answer(question)

print("Retrieved Documents:")
for d in docs:
    print("-", d)

print("\nGenerated Answer:")
print(response)